# 03 -- Celestial Geometry

This notebook covers Sun and Earth local-frame vectors, azimuth/elevation
angles, body path plots against terrain horizons, and a synthetic lightmap
generated from explicit vectors.  It draws from command-line examples 11
through 13.

**Requirements:** SPICE kernel download on first use (network access) and
the synthetic horizon bundle (downloaded automatically, ~37 MB).

**Run the individual scripts:** `11_spice_vectors.py`,
`12_body_and_horizon_plots.py`, `13_synthetic_lightmap.py`

## Setup

In [ ]:
import sys, os
from pathlib import Path

def _repo_root():
    """Find the Lunarscout repository root from the kernel working directory."""
    for start in [Path.cwd()] + list(Path.cwd().parents):
        if (start / "src" / "lunarscout" / "__init__.py").exists():
            return start
    raise RuntimeError(
        "Cannot locate Lunarscout repository root. "
        "Launch Jupyter from the repository root directory."
    )

_REPO = _repo_root()
sys.path.insert(0, str(_REPO / "src"))
sys.path.insert(0, str(_REPO / "examples"))


from datetime import timedelta

import lunarscout as ls
import numpy as np
import matplotlib.pyplot as plt

from _example_support import ensure_synthetic_horizon_scenario

WORKSPACE = Path("/tmp/lunarscout_notebook_03")
WORKSPACE.mkdir(parents=True, exist_ok=True)

# Choose a south-polar point (must be within the synthetic horizon DEM).
# lat=-89.99 is ~300m from the pole; the 256x256, 20m-pixel DEM covers ~5km.
POINT = ls.LonLat(longitude=0.0, latitude=-89.99)

times_short = list(ls.iter_times(
    "2027-01-01T00:00:00Z", "2027-01-01T06:00:00Z", timedelta(hours=2),
))
times_long = list(ls.iter_times(
    "2027-01-01T00:00:00Z", "2027-01-02T00:00:00Z", timedelta(hours=1),
))

print(f"South-polar point: {POINT}")
print(f"Short sample: {len(times_short)} times  (elevation plots)")
print(f"Long sample:  {len(times_long)} times   (horizon paths)")

---
## 1. SPICE Vectors and Angles

Vectors are returned in the local North-East-Down frame, in km.  Angles use
azimuth 0 = north, 90 = east; elevation +90 = straight up.

The first call downloads and caches default SPICE kernels.  Subsequent runs
reuse the cache.

In [ ]:
vectors_ned = ls.body_vectors_ned(POINT, "sun", times_short)
angles = ls.body_azimuth_elevation(POINT, "sun", times_short)

print("Sun NED vectors (km) -- shape:", vectors_ned.shape)
for i, t in enumerate(times_short):
    v = vectors_ned[i]
    a = angles[i]
    print(f"  {t}: NED=({v[0]:+.1f}, {v[1]:+.1f}, {v[2]:+.1f})  "
          f"az={a[0]:.0f} deg  el={a[1]:.1f} deg")

In [ ]:
# Earth geometry as DataFrames.
df = ls.body_azimuth_elevation_dataframe(POINT, "earth", times_short)
df

**Try this:** change the point to `LonLat(longitude=45.0, latitude=-85.0)`
and compare the Sun azimuth progression.

---
## 2. Body Elevation Over Time

In [ ]:
fig, ax = ls.plot_body_elevation(POINT, "sun", times_long, grid=True)
fig.suptitle("Sun Elevation vs Time")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = ls.plot_body_elevations(
    POINT, bodies=["sun", "earth"],
    times=times_long, grid=True,
)
fig.suptitle("Sun and Earth Elevation")
fig.tight_layout()
plt.show()

**Try this:** set `grid=False` and experiment with the Matplotlib axes
returned by the plot functions.

---
## 3. Terrain Horizon and Body Paths

This section requires the synthetic horizon scenario (downloaded
automatically).  The conceptual shift is from elevation relative to an
ideal horizontal plane to elevation relative to actual surrounding terrain.

In [ ]:
scenario = ensure_synthetic_horizon_scenario(WORKSPACE)
if scenario is None:
    print("Skipping horizon section -- synthetic horizon data not available.")
    print("Run again with network access for the initial download.")
else:
    print(f"Horizon scenario root: {scenario.root_path()}")

In [ ]:
if scenario is not None:
    fig, ax = scenario.plot_horizon(POINT, center_azimuth=0.0)
    fig.suptitle("Terrain Horizon (north-centred)")
    fig.tight_layout()
    plt.show()

In [ ]:
if scenario is not None:
    fig, ax = scenario.plot_horizon(POINT, center_azimuth=90.0)
    scenario.plot_body_path(ax, POINT, body="sun", times=times_long,
                            style="center_and_limbs", label="Sun")
    scenario.plot_body_path(ax, POINT, body="earth", times=times_long,
                            style="limbs", label="Earth")
    ax.legend()
    fig.suptitle("Sun and Earth Paths over Horizon")
    fig.tight_layout()
    plt.show()

In [ ]:
if scenario is not None:
    fig, ax = scenario.plot_zoomed_body_path(
        POINT, bodies=["sun", "earth"], times=times_long,
        observer_height_decimeters=0,
    )
    fig.suptitle("Zoomed Body Path over Horizon")
    fig.tight_layout()
    plt.show()

**Try this:** change `center_azimuth` to 180 or 270 and re-plot the
horizon.  Use `over_horizon=True` with `plot_body_elevations` to see
elevation above the terrain.

---
## 4. Synthetic Lightmap with Explicit Vectors

Generate a lightmap using explicit Moon-ME Sun vectors.  This avoids SPICE
kernel loading during the product call and runs on CPU.

In [ ]:
if scenario is not None:
    from datetime import datetime, timezone

    output = WORKSPACE / "horizon_scenario"
    lightmap_path = output / "analysis" / "synthetic_lightmap.tif"
    lightmap_path.parent.mkdir(parents=True, exist_ok=True)

    # Hard-coded Sun vectors for reproducibility (km, Moon-ME).
    sun_distance_km = 147_010_225.48
    sun_vecs = np.array([
        [sun_distance_km, 0.0, 0.0],
        [sun_distance_km, 0.0, 0.0],
        [sun_distance_km, 0.0, 0.0],
        [sun_distance_km, 0.0, 0.0],
    ], dtype=np.float64)

    ls.generate_lightmap(
        str(output / "dem.tif"),
        str(output / "horizons"),
        str(lightmap_path),
        times=[
            datetime(2027, 1, 1, h, tzinfo=timezone.utc)
            for h in range(0, 8, 2)
        ],
        sun_vectors_m=sun_vecs,
        backend="cpu",
        overwrite=True,
    )
    print(f"Lightmap written to {lightmap_path}")

    # Read back band statistics.
    for b in range(4):
        arr, _ = ls.read_geotiff(lightmap_path, band=b + 1)
        masked = arr[arr != 0]
        print(f"  Band {b}: {len(masked)} valid pixels, "
              f"range [{masked.min()}, {masked.max()}]")

**Try this:** vary the Sun vector direction to simulate different
illumination geometries.  The lightmap encodes visible solar fraction as
`uint8` (0 = fully obscured, 255 = fully visible).